In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

# =====================================================
# PARAMETER
# =====================================================
dbutils.widgets.text("source_table", "")
source_table = dbutils.widgets.get("source_table")

pipeline_run_id = str(uuid.uuid4())

print(f"Processing Silver Layer for: {source_table}")

# =====================================================
# TARGET CONFIG
# =====================================================
target_table = source_table.replace("bronze", "silver")

catalog, schema, table_name = target_table.split(".")

# =====================================================
# CREATE CATALOG + SCHEMA
# =====================================================
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# =====================================================
# READ BRONZE
# =====================================================
df = spark.table(source_table)

# =====================================================
# BASE METADATA COLUMNS (EXCLUDED FROM HASH)
# =====================================================
metadata_columns = [
    "created_at", "updated_at", "silver_ingested_at",
    "pipeline_run_id", "is_active",
    "migrated_at", "ingestion_date",
    "batch_id", "source_table", "source_system",
    "load_start_id", "load_end_id",
    "uniq_id", "record_hash"
]

# =====================================================
# ================= TRANSFORMATIONS ===================
# =====================================================

# ---------------- CUSTOMERS ----------------
if table_name == "customers":
    df = df.withColumns({
        "first_name": initcap(trim(col("first_name"))),
        "last_name": initcap(trim(col("last_name"))),
        "email": lower(col("email")),
        "phone": concat(lit("+91"), col("phone")),
        "age": floor(months_between(current_date(), col("date_of_birth")) / 12)
    })

    df = df.withColumn(
        "customer_segment",
        when(col("annual_income") > 100000, "PREMIUM")
        .when(col("annual_income") > 50000, "MID")
        .otherwise("BASIC")
    )

# ---------------- ADDRESSES ----------------
elif table_name == "addresses":
    df = df.withColumns({
        "city": initcap(col("city")),
        "state": upper(col("state"))
    })

    df = df.withColumn(
        "region",
        when(col("state").isin("MH","DL","KA"), "WEST")
        .otherwise("OTHER")
    )

# ---------------- AGENTS ----------------
elif table_name == "agents":
    df = df.withColumns({
        "agent_name": initcap(trim(col("agent_name"))),
        "agency_name": initcap(trim(col("agency_name")))
    })

# ---------------- APPLICATIONS ----------------
elif table_name == "applications":
    df = df.withColumn(
        "risk_band",
        when(col("coverage_amount") >= 200000, "HIGH")
        .when(col("coverage_amount") >= 100000, "MEDIUM")
        .otherwise("LOW")
    )

# ---------------- POLICIES ----------------
elif table_name == "policies":
    df = df.withColumn(
        "policy_tenure_days",
        datediff(col("end_date"), col("start_date"))
    ).withColumn(
        "is_active_policy",
        when(col("status") == "ACTIVE", True).otherwise(False)
    )

# ---------------- POLICY RIDERS ----------------
elif table_name == "policy_riders":
    df = df.withColumn(
        "rider_category",
        when(col("rider_amount") > 10000, "HIGH_VALUE")
        .otherwise("STANDARD")
    )

# ---------------- BENEFICIARIES ----------------
elif table_name == "beneficiaries":
    df = df.withColumn(
        "is_primary_beneficiary",
        when(col("allocation_percent") >= 50, True).otherwise(False)
    )

# ---------------- MEDICAL RECORDS ----------------
elif table_name == "medical_records":
    df = df.withColumn(
        "health_risk_flag",
        when(col("bmi") >= 30, "HIGH_RISK")
        .when(col("bmi") >= 25, "MEDIUM_RISK")
        .otherwise("LOW_RISK")
    )

# ---------------- BILLING ACCOUNTS ----------------
elif table_name == "billing_accounts":
    df = df.withColumn(
        "billing_type",
        when(col("autopay_enabled") == True, "AUTOPAY")
        .otherwise("MANUAL")
    )

# ---------------- PREMIUM PAYMENTS ----------------
elif table_name == "premium_payments":
    df = df.withColumn(
        "payment_status",
        upper(col("payment_status"))
    ).withColumn(
        "is_late_payment",
        when(col("days_late") > 0, True).otherwise(False)
    )

# ---------------- CLAIMS ----------------
elif table_name == "claims":
    df = df.withColumn(
        "claim_status_category",
        when(col("status") == "APPROVED", "SETTLED")
        .when(col("status") == "REJECTED", "DENIED")
        .otherwise("PENDING")
    )

# ---------------- DOCUMENTS ----------------
elif table_name == "documents":
    df = df.withColumn(
        "document_class",
        when(col("file_size_bytes") > 5000000, "LARGE")
        .otherwise("NORMAL")
    )

# ---------------- AUDIT LOG ----------------
elif table_name == "audit_log":
    df = df.withColumn(
        "operation_type",
        upper(col("operation"))
    )

# ---------------- POLICY DIVIDENDS ----------------
elif table_name == "policy_dividends":
    df = df.withColumn(
        "dividend_performance",
        when(col("declared_amount") > col("applied_amount"), "GAIN")
        .otherwise("LOSS")
    )

# ---------------- UNDERWRITING ----------------
elif table_name == "underwriting_assessments":
    df = df.withColumns({
        "bmi_category": when(col("bmi") >= 30, "OBESE")
                        .when(col("bmi") >= 25, "OVERWEIGHT")
                        .otherwise("NORMAL")
    })

# =====================================================
# METADATA ENRICHMENT
# =====================================================

df = df.withColumns({
    "created_at": current_timestamp(),
    "updated_at": current_timestamp(),
    "silver_ingested_at": current_timestamp(),
    "pipeline_run_id": lit(pipeline_run_id),
    "is_active": lit(True),
    "migrated_at": current_timestamp(),
    "ingestion_date": to_date(current_timestamp()),
    "batch_id": lit(f"{source_table}"),
    "source_table": lit(source_table),
    "source_system": lit(schema),
    "load_start_id": lit("0"),
    "load_end_id": lit("999999999")
})

# =====================================================
# RECORD HASH (CDC)
# =====================================================

business_cols = [c for c in df.columns if c not in metadata_columns]

df = df.withColumn(
    "record_hash",
    sha2(concat_ws("||", *business_cols), 256)
)

# =====================================================
# DEDUPLICATION
# =====================================================
df = df.dropDuplicates(["record_hash"])

# =====================================================
# WRITE TO SILVER
# =====================================================

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"SUCCESS: Loaded Silver Table -> {target_table}")